# 工具搜索：替代方案

**建议先阅读另一篇 Cookbook：Tool Search with Embeddings。**

本篇 Cookbook 的目标是展示在 Claude 上使用工具搜索（更准确地说是“工具发现”）的几种替代方案。我们将演示两种实用技巧：

1. 工具可以在没有“搜索”的情况下被发现。在本 Cookbook 中，我们把全部工具名称写入 Claude 的系统提示词，并为 Claude 提供 decribe_tool_tool，用于把工具完整加载到 
Claude 的上下文中。
2. 如果工具尚未加载到 Claude 的上下文中，就不必把它们放进请求的 `tools` 列表。这会增加一些应用侧的复杂度，但可以让你的
应用即使面对 Claude 可能访问的数千个工具，也依然保持请求体积小巧。

用户在设计工具搜索时拥有很大的自由度，可以让 Claude 的上下文（以及 Messages 请求）尽可能聚焦。

## 前置条件

在跟随本指南操作之前，请确认你已具备：

**必备知识**
- Python 基础——熟悉函数、字典和基本数据结构
- 对 Claude 工具使用的基本理解——建议先阅读[工具使用指南](https://docs.anthropic.com/en/docs/build-with-claude/tool-use)

**必备工具**
- Python 3.11 或更高版本
- Anthropic API 密钥（[在此获取](https://docs.anthropic.com/claude/reference/getting-started-with-the-api)）

In [ ]:
%pip install -q anthropic python-dotenv

In [ ]:
import json

import anthropic
from dotenv import load_dotenv

load_dotenv()

MODEL = "claude-sonnet-4-6"
client = anthropic.Anthropic()

print("✓ Client initialized")

## 定义工具库

我们定义 5 个简单的工具。在生产环境中，这里可能是存放在数据库或配置文件中的数百乃至数千个工具。

In [ ]:
TOOL_LIBRARY = {
    "get_weather": {
        "name": "get_weather",
        "description": "Get current weather for a city",
        "input_schema": {
            "type": "object",
            "properties": {
                "city": {"type": "string", "description": "City name"},
            },
            "required": ["city"],
        },
    },
    "get_stock_price": {
        "name": "get_stock_price",
        "description": "Get current stock price for a ticker symbol",
        "input_schema": {
            "type": "object",
            "properties": {
                "ticker": {"type": "string", "description": "Stock ticker (e.g., AAPL)"},
            },
            "required": ["ticker"],
        },
    },
    "convert_currency": {
        "name": "convert_currency",
        "description": "Convert amount between currencies",
        "input_schema": {
            "type": "object",
            "properties": {
                "amount": {"type": "number"},
                "from_currency": {"type": "string"},
                "to_currency": {"type": "string"},
            },
            "required": ["amount", "from_currency", "to_currency"],
        },
    },
    "calculate_tip": {
        "name": "calculate_tip",
        "description": "Calculate tip amount for a bill",
        "input_schema": {
            "type": "object",
            "properties": {
                "bill_amount": {"type": "number"},
                "tip_percent": {"type": "number", "default": 20},
            },
            "required": ["bill_amount"],
        },
    },
    "send_email": {
        "name": "send_email",
        "description": "Send an email to a recipient",
        "input_schema": {
            "type": "object",
            "properties": {
                "to": {"type": "string", "description": "Recipient email"},
                "subject": {"type": "string"},
                "body": {"type": "string"},
            },
            "required": ["to", "subject", "body"],
        },
    },
}

print(f"✓ Defined {len(TOOL_LIBRARY)} tools: {list(TOOL_LIBRARY.keys())}")

## `describe_tool` 工具

我们不使用语义搜索，而是给 Claude 一个简单的 `describe_tool` 工具。Claude 传入工具名称调用它，即可把该工具加载到上下文中。

系统提示词中列出了所有可用工具的名称，因此无需嵌入向量或搜索，Claude 就知道有哪些工具可用。

In [ ]:
DESCRIBE_TOOL = {
    "name": "describe_tool",
    "description": "Load a tool's full definition into context. Call this before using any tool for the first time.",
    "input_schema": {
        "type": "object",
        "properties": {
            "tool_name": {
                "type": "string",
                "description": "Name of the tool to load",
            },
        },
        "required": ["tool_name"],
    },
}

# Build system prompt with tool catalog
tool_names = list(TOOL_LIBRARY.keys())
SYSTEM_PROMPT = f"""You are a helpful assistant with access to various tools.

Available tools: {", ".join(tool_names)}

Before using any tool, you must first call describe_tool with the tool name to load it."""

print("System prompt:")
print(SYSTEM_PROMPT)

## 模拟工具执行

为演示目的提供简单的模拟响应：

In [ ]:
def execute_tool(name: str, inputs: dict) -> str:
    """Mock tool execution."""
    if name == "get_weather":
        return json.dumps({"city": inputs["city"], "temp": "72°F", "conditions": "Sunny"})
    elif name == "get_stock_price":
        return json.dumps({"ticker": inputs["ticker"], "price": 185.50, "change": "+1.2%"})
    elif name == "convert_currency":
        rate = 0.92 if inputs["to_currency"] == "EUR" else 1.0
        converted = inputs["amount"] * rate
        return json.dumps({"converted": round(converted, 2), "to": inputs["to_currency"]})
    elif name == "calculate_tip":
        tip = inputs["bill_amount"] * (inputs.get("tip_percent", 20) / 100)
        return json.dumps({"tip": round(tip, 2), "total": round(inputs["bill_amount"] + tip, 2)})
    elif name == "send_email":
        return json.dumps({"status": "sent", "to": inputs["to"]})
    return json.dumps({"error": f"Unknown tool: {name}"})


print("✓ Mock execution ready")

## 带动态工具加载的对话循环

这里的关键模式是：

1. **工具列表中一开始只包含 `describe_tool`**
2. **当 Claude 调用 `describe_tool` 时**，返回一个 `tool_reference`，同时用 `defer_loading=True` 把该工具加入 `active_tools`
3. **`defer_loading=True` 至关重要**——它让工具定义不进入已缓存的提示词前缀，从而在发现新工具时避免缓存失效

每当 Claude 在对话中看到 `tool_reference`，完整的工具定义就会在对话的那个位置加载到 Claude 的上下文中。

In [ ]:
def run_conversation(user_message: str, max_turns: int = 10):
    """Run a conversation with dynamic tool loading."""
    print(f"\n{'=' * 60}")
    print(f"USER: {user_message}")
    print(f"{'=' * 60}\n")

    messages = [{"role": "user", "content": user_message}]

    # Start with ONLY describe_tool - no other tools in the request
    active_tools = [DESCRIBE_TOOL]
    loaded_tools = set()  # Track which tools we've added

    for turn in range(max_turns):
        print(f"--- Turn {turn + 1} (tools in request: {len(active_tools)}) ---")

        response = client.messages.create(
            model=MODEL,
            max_tokens=1024,
            system=SYSTEM_PROMPT,
            tools=active_tools,
            messages=messages,
            extra_headers={"anthropic-beta": "advanced-tool-use-2025-11-20"},
        )

        messages.append({"role": "assistant", "content": response.content})

        if response.stop_reason == "end_turn":
            for block in response.content:
                if block.type == "text":
                    print(f"\nASSISTANT: {block.text}")
            break

        # Process tool calls
        tool_results = []
        for block in response.content:
            if block.type == "text" and block.text:
                print(f"ASSISTANT: {block.text}")

            elif block.type == "tool_use":
                tool_name = block.name
                tool_input = block.input

                if tool_name == "describe_tool":
                    requested_tool = tool_input["tool_name"]
                    print(f"🔍 describe_tool({requested_tool})")

                    if requested_tool in TOOL_LIBRARY:
                        # Add tool to active_tools with defer_loading=True
                        # This is critical for prompt caching!
                        if requested_tool not in loaded_tools:
                            tool_def = {**TOOL_LIBRARY[requested_tool], "defer_loading": True}
                            active_tools.append(tool_def)
                            loaded_tools.add(requested_tool)
                            print(f"   ✓ Added {requested_tool} to tools (defer_loading=True)")

                        # Return tool_reference so Claude can use it
                        tool_results.append(
                            {
                                "type": "tool_result",
                                "tool_use_id": block.id,
                                "content": [
                                    {"type": "tool_reference", "tool_name": requested_tool}
                                ],
                            }
                        )
                    else:
                        tool_results.append(
                            {
                                "type": "tool_result",
                                "tool_use_id": block.id,
                                "content": f"Tool '{requested_tool}' not found.",
                            }
                        )
                else:
                    # Execute discovered tool
                    print(f"🔧 {tool_name}({json.dumps(tool_input)})")
                    result = execute_tool(tool_name, tool_input)
                    print(f"   → {result}")
                    tool_results.append(
                        {
                            "type": "tool_result",
                            "tool_use_id": block.id,
                            "content": result,
                        }
                    )

        if tool_results:
            messages.append({"role": "user", "content": tool_results})

    print(f"\n{'=' * 60}\n")


print("✓ Conversation loop ready")

## 示例：天气查询

观察 Claude 的行为：
1. 在系统提示词的工具列表中看到 `get_weather`
2. 调用 `describe_tool("get_weather")` 把它加载进来
3. 收到 `tool_reference`，此后即可使用该工具

In [ ]:
run_conversation("What's the weather in Tokyo?")

## 示例：多工具查询

Claude 可以按需加载多个工具：

In [ ]:
run_conversation("Convert $100 to EUR, then calculate a 20% tip on a $85 dinner bill.")

## 为什么 `defer_loading=True` 很重要

当你把工具加入 `tools` 列表时，工具定义通常会被加载到 Claude 上下文窗口的最开头。如果新增工具，上下文窗口的最开头就发生了变化，你会失去大部分缓存。

**使用 `defer_loading=True` 时：**
- 工具定义不会被放入 Claude 上下文窗口的开头。
- 取而代之的是，当 Claude 看到 `tool_reference` 时才把它加载到上下文。
- 这意味着即使不断发现新工具，你的系统提示词和初始工具仍能保持缓存命中。

**该模式的做法：**
```python
# Initial request - only describe_tool, system prompt is cached
tools = [DESCRIBE_TOOL]

# After Claude calls describe_tool("get_weather")
# Add with defer_loading to preserve cache
tools.append({**TOOL_LIBRARY["get_weather"], "defer_loading": True})

# Return tool_reference so Claude knows it's available
tool_result = [{"type": "tool_reference", "tool_name": "get_weather"}]
```

对于拥有数百或数千个工具的应用，如果你希望做到以下几点，这一做法必不可少：
- 保持初始请求体积小
- 在多次工具发现之间保持提示词缓存
- 只加载 Claude 真正需要的工具

## 结论

本 Cookbook 的核心洞见是：**在 Claude 需要某个工具之前，它不必出现在 `tools` 列表中**。把这一点与 `defer_loading=True` 和 `tool_reference` 结合，你就能扩展到数千个工具，同时保持请求体积小并保留提示词缓存。不过在这种情况下，你的客户端需要提供一套工具发现机制。

这里展示的 `describe_tool` 方案只是一种变体。其他模式包括：
- **`list_tools`**——返回匹配某个类别或关键词的工具名称
- **分层发现**——先浏览工具类别，再加载具体工具
- **混合方案**——对大型工具目录，把列表方式与语义搜索结合

核心模式始终是：
1. 发现工具时返回 `tool_reference`
2. 以 `defer_loading=True` 添加该工具，以保留缓存
3. Claude 随即可立即使用该工具

基于嵌入向量的方案请参见 [Tool Search with Embeddings](./tool_search_with_embeddings.ipynb)。